In [ ]:
import hashlib,importlib.metadata,json,os,subprocess,sys,time,urllib.request
from pathlib import Path
ROOT=Path('/kaggle/working');WORK=Path('/tmp')/('mgbfs-native-fault-'+str(os.getpid()));WORK.mkdir()
BASE='https://github.com/TryDotAtwo/MultiGPUBFS/releases/download/native-pip-20261017-v23/'
m=json.loads(urllib.request.urlopen(BASE+'manifest.json',timeout=60).read());assert m['size']<2_000_000
assert m['sha256']=='69d6b88b0c431b95d58ecc276aad38796838abf90d6cb4854eb9f85efb6e4b69' and m['source_commit']=='c7217ee8397bb71aeae811e12723fd7a2efaea73'
wheel=WORK/m['wheel'];urllib.request.urlretrieve(BASE+m['wheel'],wheel);assert hashlib.sha256(wheel.read_bytes()).hexdigest()==m['sha256']
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--force-reinstall',str(wheel)],check=True)
subprocess.run([sys.executable,'-m','pip','install','cayleypy==0.2.0'],check=True)
assert importlib.metadata.version('multigpubfs')=='0.2.0.dev23'
hardware=subprocess.check_output(['nvidia-smi','--query-gpu=name,compute_cap,driver_version','--format=csv,noheader'],text=True).strip().splitlines()
assert len(hardware)==2 and all(', 7.5,' in x for x in hardware),hardware
os.environ['MGBFS_NATIVE_CACHE']=str(WORK/'cache')
import multigpubfs
prefetch=multigpubfs.prefetch_native(sm=[75]);assert prefetch['status']=='PREFETCHED'
os.environ['MGBFS_NATIVE_OFFLINE']='1'
from multigpubfs.native_distribution import native_runtime
native,env=native_runtime();native_sha=hashlib.sha256(Path(native).read_bytes()).hexdigest()
assert native_sha=='6c579b686a945305b2ea1a54ed1d50806696c9e4c3cd22ad6224bca594127b01'
manifest=json.loads((Path(native).parent.parent/'manifest.json').read_text());assert manifest['native_failure_context_schema']==1
receipt={'schema':1,'status':'RUNNING','package':'0.2.0.dev23','wheel_sha256':m['sha256'],'package_source_commit':m['source_commit'],'native_binary_sha256':native_sha,'native_source_commit':manifest['native_binary_source_commit'],'hardware':hardware,'checks':[],
 'scope':'Physical 2xT4 normal exact BFS plus deliberately injected API failure; does not reproduce or fix the unknown production 8-GPU CUDA 999 cause'}
def record(name,**data):
 receipt['checks'].append(dict(name=name,status='PASS',**data));(ROOT/'native-fault-acceptance.json').write_text(json.dumps(receipt,indent=2));print('FAULT_CHECK',json.dumps(receipt['checks'][-1]),flush=True)
started=time.monotonic()
try:
 from multigpubfs.deep_diagnostics import bfs_probe
 for fixture in ['permutation','repeated','matrix']:
  root=WORK/('normal-'+fixture);root.mkdir()
  result=bfs_probe({'fixture':fixture,'devices':[0,1],'persistent':True},root)
  record('exact-two-rank-persistent-'+fixture,result=result)
 # A test-only LD_PRELOAD wrapper produces an error at one reporting API.
 # No fault injection feature is added to the shipped package or native engine.
 c=WORK/'inject.c';c.write_text('#define _GNU_SOURCE\n#include <dlfcn.h>\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\ntypedef int (*copy_fn)(void*,const void*,size_t,int,void*);\nint cudaMemcpyAsync(void *dst,const void *src,size_t bytes,int kind,void *stream) {\n const char *flag=getenv("MGBFS_TEST_FAIL_NATIVE"),*rank=getenv("RANK"),*path=getenv("MGBFS_JOB_PHASE_PATH");\n if(flag && rank && path && !strcmp(flag,"1") && !strcmp(rank,"1")) {\n  FILE *f=fopen(path,"r");if(f){char text[8192];size_t n=fread(text,1,sizeof(text)-1,f);fclose(f);text[n]=0;\n   if(strstr(text,"\\"phase\\":\\"bfs\\"")) return 999;\n  }\n }\n copy_fn real=(copy_fn)dlsym(RTLD_NEXT,"cudaMemcpyAsync");return real(dst,src,bytes,kind,stream);\n}\n')
 injector=WORK/'inject.so';subprocess.run(['gcc','-shared','-fPIC','-O2','-o',str(injector),str(c),'-ldl'],check=True)
 child=WORK/'fault-child.py';child.write_text("import contextlib,json,os,sys\nfrom pathlib import Path\nfrom multigpubfs.native_distribution import native_runtime\nfrom multigpubfs.generic_session import persistent_graph_workers\nfrom multigpubfs.graph_definition import GraphDefinition\nfrom multigpubfs.launch import run_graph\nmode,output,result=sys.argv[1:];output=Path(output)\nnative,env=native_runtime();env.update(MGBFS_TRACE_NATIVE_CALLS='1',MGBFS_TEST_FAIL_NATIVE='1')\ngraph=GraphDefinition.permutation([[1,0,2],[0,2,1]],[0,1,2])\nworker_root=None\ntry:\n with persistent_graph_workers() if mode=='persistent' else contextlib.nullcontext() as pool:\n  if pool is not None:worker_root=pool.root\n  run_graph(graph,output,devices=[0,1],capacity=64,shards=1,autotune=False,max_seconds=20,_batch=8,_native_env=env)\n raise AssertionError('Expected native failure was not triggered')\nexcept RuntimeError as error:\n assert 'NATIVE_DISTRIBUTED_RANK_FAILED' in str(error),str(error)\n assert 'evidence=' in str(error)\nevidence=output/'failure-evidence';summary=json.loads((evidence/'summary.json').read_text())\nassert [c['when'] for c in summary['captures']]==['before_cleanup','after_cleanup']\nphase=json.loads((evidence/'rank-1/phase.json').read_text());fault=phase['native_error']\nassert fault['code']==999 and fault['call']=='cudaMemcpyAsync' and fault['rank']==1 and fault['device']==1,fault\nassert fault['layer']==1 and fault['last_completed_layer']==0,fault\nassert fault['source'].endswith('generic_distributed_native.rs'),fault\nassert len(summary['captures'][-1]['ranks'])==2\nassert 'MGBFS_NATIVE_ERROR' in (evidence/'rank-1/worker.log').read_text()\nassert worker_root is None or not worker_root.exists()\npayload={'status':'PASS','mode':mode,'fault':fault,'captured_ranks':2,'captures':['before_cleanup','after_cleanup'],'files_survived_worker_cleanup':True,'no_restart':True}\nPath(result).write_text(json.dumps(payload,indent=2));print(json.dumps(payload),flush=True)\n")
 for mode in ['persistent','oneshot']:
  output=WORK/('injected-'+mode);result=ROOT/('injection-'+mode+'.json')
  proc=subprocess.run([sys.executable,str(child),mode,str(output),str(result)],env=dict(os.environ,LD_PRELOAD=str(injector)),text=True,capture_output=True,timeout=90)
  (ROOT/('injection-'+mode+'.log')).write_text(proc.stdout+'\n'+proc.stderr)
  print(proc.stdout[-6000:],proc.stderr[-6000:],flush=True)
  assert proc.returncode==0,(mode,proc.returncode)
  data=json.loads(result.read_text());assert data['status']=='PASS';record('injected-999-'+mode,**{k:v for k,v in data.items() if k!='status'})
 receipt.update(status='PASS',wall_seconds=time.monotonic()-started);(ROOT/'native-fault-acceptance.json').write_text(json.dumps(receipt,indent=2));print('NATIVE_FAILURE_CONTEXT_GPU_ACCEPTANCE_PASS',flush=True)
except BaseException as error:
 receipt.update(status='FAIL',error=str(error),exception_type=type(error).__name__);(ROOT/'native-fault-acceptance.json').write_text(json.dumps(receipt,indent=2));raise
